## WMATA DATA INTEGRATION FEASIBILITY

01. WMATA and GTFS station code matching: 92.16%
02. WMATA and weather timestamp matching: 100%

In [29]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile
import io

In [18]:
from pathlib import Path

PROJECT = Path.home() / "Downloads" / "602DS_Fall2026_Group2-backup"
#PROJECT = Path.cwd()   change the directory if you want to use it!!

if not (PROJECT / "WMATA_12h_pilot.zip").exists():
    PROJECT = PROJECT.parent

ZIP_PATH = PROJECT / "WMATA_12h_pilot.zip"

print("Project exists:", PROJECT.exists())
print("ZIP exists:", ZIP_PATH.exists())

Project exists: True
ZIP exists: True


In [19]:
with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    files = zip_file.namelist()

for file in files:
    print(file)

data/
data/processed/
data/processed/.gitkeep
data/raw/
data/raw/gtfs_rt/
data/raw/gtfs_rt/vehiclepositions/
data/raw/gtfs_rt/vehiclepositions/2026-10-23/
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T213433Z.pb.gz
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T211912Z.pb.gz
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T224111Z.pb.gz
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T202040Z.pb.gz
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T211607Z.pb.gz
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T232517Z.pb.gz
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T232417Z.pb.gz
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T201939Z.pb.gz
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T191701Z.pb.gz
data/raw/gtfs_rt/vehiclepositions/2026-10-23/vehiclepositions_20261023T223908Z.p

In [20]:
keywords = ["predictions", "weather", "stops", "gtfs"]

for file in files:
    if any(word in file.lower() for word in keywords):
        if not file.endswith(".pb.gz"):
            print(file)

data/raw/gtfs_rt/
data/raw/gtfs_rt/vehiclepositions/
data/raw/gtfs_rt/vehiclepositions/2026-10-23/
data/raw/gtfs_rt/vehiclepositions/2026-10-24/
data/raw/gtfs_rt/alerts/
data/raw/gtfs_rt/alerts/2026-10-23/
data/raw/gtfs_rt/alerts/2026-10-24/
data/raw/gtfs_rt/tripupdates/
data/raw/gtfs_rt/tripupdates/2026-10-23/
data/raw/gtfs_rt/tripupdates/2026-10-24/
data/raw/weather_2026-10-24.csv
data/raw/wmata_gtfs_2026-10-23.zip
data/raw/weather_2026-10-23.csv
data/raw/wmata_predictions_2026-10-24.csv
data/raw/wmata_predictions_2026-10-23.csv


In [28]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    predictions = pd.concat([
        pd.read_csv(z.open("data/raw/wmata_predictions_2026-10-23.csv")),
        pd.read_csv(z.open("data/raw/wmata_predictions_2026-10-24.csv"))
    ], ignore_index=True)

    weather = pd.concat([
        pd.read_csv(z.open("data/raw/weather_2026-10-23.csv")),
        pd.read_csv(z.open("data/raw/weather_2026-10-24.csv"))
    ], ignore_index=True)

print("WMATA shape:", predictions.shape)
print("Weather shape:", weather.shape)

WMATA shape: (752791, 10)
Weather shape: (97, 12)


In [22]:
print("WMATA columns:")
print(predictions.columns.tolist())

print("\nWeather columns:")
print(weather.columns.tolist())

WMATA columns:
['Car', 'Destination', 'DestinationCode', 'DestinationName', 'Group', 'Line', 'LocationCode', 'LocationName', 'Min', 'snapshot_ts']

Weather columns:
['time', 'interval', 'temperature_2m', 'precipitation', 'rain', 'snowfall', 'wind_speed_10m', 'wind_gusts_10m', 'weather_code', 'snapshot_ts', 'latitude', 'longitude']


In [30]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    gtfs_data = z.read("data/raw/wmata_gtfs_2026-10-23.zip")

with zipfile.ZipFile(io.BytesIO(gtfs_data), "r") as gtfs:
    print(gtfs.namelist())

['agency.txt', 'calendar_dates.txt', 'routes.txt', 'shapes.txt', 'stop_times.txt', 'stops.txt', 'trips.txt', 'feed_info.txt', 'levels.txt', 'pathways.txt']


In [24]:
with zipfile.ZipFile(io.BytesIO(gtfs_data), "r") as gtfs:
    stops = pd.read_csv(gtfs.open("stops.txt"))

print("GTFS shape:", stops.shape)
print("\nGTFS columns:")
print(stops.columns.tolist())

stops.head()

GTFS shape: (2176, 10)

GTFS columns:
['stop_id', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'location_type', 'parent_station', 'wheelchair_boarding', 'level_id']


,stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,location_type,parent_station,wheelchair_boarding,level_id
0,STN_N06,Wiehle-Reston East,NaN,38.947832,-77.340316,91,1,NaN,1,NaN
1,STN_N04,Spring Hill,NaN,38.929212,-77.241891,90,1,NaN,1,NaN
2,STN_N03,Greensboro,NaN,38.921269,-77.234162,89,1,NaN,1,NaN
3,STN_N02,Tysons,NaN,38.920685,-77.221589,88,1,NaN,1,NaN
4,STN_N01,McLean,NaN,38.924339,-77.210530,87,1,NaN,1,NaN


## 01. WMATA and GTFS Station Code Matching

In [25]:
wmata_codes = set(predictions["LocationCode"].dropna().astype(str))

gtfs_stations = stops[stops["location_type"] == 1].copy()

gtfs_stations["station_code"] = gtfs_stations["stop_id"].str.replace(
    "STN_", "", regex=False
)

gtfs_codes = set(gtfs_stations["station_code"].dropna())

matched = wmata_codes.intersection(gtfs_codes)
unmatched = wmata_codes.difference(gtfs_codes)

print("WMATA stations:", len(wmata_codes))
print("GTFS stations:", len(gtfs_codes))
print("Matched stations:", len(matched))
print("Matching rate:", round(len(matched) / len(wmata_codes) * 100, 2), "%")
print("Unmatched WMATA codes:", sorted(unmatched))

WMATA stations: 102
GTFS stations: 98
Matched stations: 94
Matching rate: 92.16 %
Unmatched WMATA codes: ['A01', 'B01', 'B06', 'C01', 'D03', 'E06', 'F01', 'F03']


## 02. WMATA and Weather Timestamp Matching

In [26]:
predictions["snapshot_ts"] = pd.to_datetime(
    predictions["snapshot_ts"], utc=True
)

weather["time"] = pd.to_datetime(
    weather["time"], utc=True
)

print("WMATA time range:")
print(predictions["snapshot_ts"].min())
print(predictions["snapshot_ts"].max())

print("\nWeather time range:")
print(weather["time"].min())
print(weather["time"].max())

WMATA time range:
2026-10-23 18:25:40.366110+00:00
2026-10-24 06:35:57.084904+00:00

Weather time range:
2026-10-23 18:15:00+00:00
2026-10-24 06:15:00+00:00


In [27]:
wmata_time = predictions.copy()
weather_time = weather.copy()

wmata_time = wmata_time.sort_values("snapshot_ts")
weather_time = weather_time.sort_values("time")

merged = pd.merge_asof(
    wmata_time,
    weather_time,
    left_on="snapshot_ts",
    right_on="time",
    direction="backward",
    tolerance=pd.Timedelta("1h")
)

matched_rows = merged["time"].notna().sum()
total_rows = len(merged)

print("Total WMATA records:", total_rows)
print("Matched records:", matched_rows)
print("Unmatched records:", total_rows - matched_rows)
print("Matching rate:", round(matched_rows / total_rows * 100, 2), "%")

Total WMATA records: 752791
Matched records: 752791
Unmatched records: 0
Matching rate: 100.0 %
